In [11]:
import os
import plotly.graph_objects as go
import gliquid.config as cfg
from gliquid.binary import BinaryLiquid, BLPlotter

# Set the environmental variable for Materials Project API key before using the BinaryLiquid class
# If using MPDS data, you will need to set the MPDS API key and ensure cfg.data_dir contains the necessary data files
os.environ["NEW_MP_API_KEY"] = "Jcw46im7UV1xOfHzbZZ8nkq8BH00Pf6s"

In [ ]:
# Instantiate a BinaryLiquid object with a guess for parameter values
bl = BinaryLiquid.from_cache('Pb-Tb', param_format='combined_no_1S')

ml_parameters = [-147186.4688,	-18.38106271,	52796.44231,	0] # from v18 9-10-25 benchmarked model
bl.update_params(ml_parameters) # update the parameters and generate the DFT-referenced phase diagram
predicted_liquidus_line = [[p[0], p[1]] for p in bl.phases[-1]['points']] # save for later plotting

blp = BLPlotter(bl)
blp.show('pred') # show the (pred)icted phase diagram
blp.show('ch+g') # show the dft convex hull (ch) with the liquid free energy (g) overlaid 

Pb: H_fusion = 4774 J/mol, T_fusion = 600.7 K, T_vaporization = 2022.15 K
Tb: H_fusion = 10790 J/mol, T_fusion = 1629 K, T_vaporization = 3503.15 K

No cached binary phase data found!
MPDS_API_KEY not found in environment variables. Proceeding without binary phase data.


No arguments specified for 't_vals', setting 't_units' to 'K'


Following this initial prediction, this system was identified as a system that our collaborators at Baylor University
were capable of conducting an assessment on using their DSC techniques.

Below is a table of measurements from their assessment of the Pb-Tb system:

## Table 2. DSC runs of the Tb-Pb system and weight % change

| Composition (%Tb: %Pb) | Solidus (±0.5°C) | Liquidus (±0.5°C) | Weight Change (±0.1%) |
|-------------------------|------------------|-------------------|----------------------|
| 10: 90                  | -                | 857.8             | 0.081                |
| 80: 20a                 | 991.4            | 1152.7            | 0.000                |
| 80: 20b                 | 991.1            | 1157.4            | 0.062                |
| 80: 20c                 | 985.8            | 1152.2            | 0.122                |
| 90: 10a                 | 990.0            | 1150.7            | 0.128                |
| 90: 10b                 | 990.7            | ~1154             | 0.079                |
| 90: 10c                 | 981.7            | ~1133.3           | 0.010                |

The letters a,b,c after the compositions are there only to distinguish samples of the same composition. The exponents are to signify samples that were prepared differently while trying to determine the preparation method that provides the most homogenous sample. 

Credit:

Mario A. Plata,<sup>a</sup> J. Streit Smith,<sup>a</sup> and Julia Y. Chan<sup>* a</sup><br>
<sup>a</sup> Department of Chemistry and Biochemistry, Baylor University, Waco, Texas 76798, United States.<br>
<sup>*</sup> Department of Chemistry and Biochemistry, Baylor University, Waco, Texas 76798, United States;<br>
https://orcid.org/0000-0003-4434-2160; Email: Julia_Chan@baylor.edu<br>


In [13]:
# composition wt% Tb, solidus temperature (C), liquidus temperature (C)
measured_points_wtpct = [[10,   None,   857.8],
                         [80,   991.4,  1152.7],
                         [80,   991.1,  1157.4],
                         [80,   985.8,  1152.2],
                         [90,   990.0,  1150.7],
                         [90,   990.7,  1154],
                         [90,   981.7,  1133.3]]

We'll need to convert these points from weight percentage to atomic percentage, and we'll take the averages for multiple measurements at the same conditions

In [14]:
def process_measured_points(measured_points_wtpct):
    """
    Average multiple measurements at the same composition.
    Return solidus points, liquidus points, and compositions with no melting.
    """
    s, l, = [], []
    s_dict, l_dict = {}, {}

    for wt_pct, solidus, liquidus in measured_points_wtpct:

        if solidus is not None:
            if wt_pct not in s_dict:
                s.append([wt_pct, 0])
            s_dict[wt_pct] = s_dict.get(wt_pct, []) + [solidus]
        if liquidus is not None:
            if wt_pct not in l_dict:
                l.append([wt_pct, 0])
            l_dict[wt_pct] = l_dict.get(wt_pct, []) + [liquidus]

    for point in s:
        point[1] = round(sum(s_dict[point[0]]) / len(s_dict[point[0]]), 1)
    for point in l:
        point[1] = round(sum(l_dict[point[0]]) / len(l_dict[point[0]]), 1)

    return s, l,

solidus_points, liquidus_points = process_measured_points(measured_points_wtpct)
print("Solidus points:", solidus_points)
print("Liquidus points:", liquidus_points)

Solidus points: [[80, 989.4], [90, 987.5]]
Liquidus points: [[10, 857.8], [80, 1154.1], [90, 1146.0]]


Now that we have some datapoints, let's add them to the figure to see how well our initial predicition performed

In [15]:
def add_experimental_traces(figure: go.Figure):
    """Add experimental observation traces to a plotly figure"""

    # Add solidus observed markers
    figure.add_trace(go.Scatter(
            x=[point[0] for point in solidus_points],
            y=[point[1] for point in solidus_points],
            mode='markers',
            marker=dict(color='#ADEBB3', size=14, symbol='triangle-up', line=dict(width=1, color='black')),
            name='Solidus Observed'
    ))

    # Add liquidus observed markers
    figure.add_trace(go.Scatter(
            x=[point[0] for point in liquidus_points],
            y=[point[1] for point in liquidus_points],
            mode='markers',
            marker=dict(color='cornflowerblue', size=12, symbol='square', line=dict(width=1, color='black')),
            name='Liquidus Observed'
    ))

fig = blp.get_plot('pred')
add_experimental_traces(fig) # add the experimental data to the plot
fig.show()

It appears that our initial prediction underestimated the melting temperatures in all compositions measured. Let's use this data to adjust the mixing parameters for the DFT-referenced liquid free energy

In [16]:
import numpy as np
from gliquid.binary import _x_vals

def optimize_predicted_params(bl: BinaryLiquid, known_points: list, predicted_params: list) -> tuple:

    hull_points = np.array([[p['comp'], p['energy']] for p in bl.phases if 'comp' in p])
    bl.eqs['h_hull_interp'] = np.interp(_x_vals[1:-1], hull_points[:, 0], hull_points[:, 1])
    bl.comp_range_fit_lim = 0
    bl.init_error = False
    bl.max_liq_temp = max(bl.component_data.values(), key=lambda x: x[1])[1]
    bl.min_liq_temp = min(bl.component_data.values(), key=lambda x: x[1])[1]
    bl.digitized_liq = [[p[0], p[1]] for p in known_points]

    mae, _ = bl.calculate_deviation_metrics(ignored_ranges=False, allow_sparse_data=True)
    print(f"Known points (Kelvin): {known_points}")
    print(f"Liquidus MAE from known points: {int(mae)}K")

    print(f"Original predicted parameters: {predicted_params}")
   
    bl.fit_parameters(verbose=True, max_iter=128, check_phase_mismatch=False, allow_sparse_data=True,
                      disable_inv_constrs=True, check_lupis_elliott=True)
    
    print(f"Adjusted parameters: {bl.get_params()}")
    return bl.get_params()


# Add a small amount of solid solubility for the a-Tb allotrope since it appears in all similar phase diagram assessments
aTb_allotrope = {'name': 'α-Tb', 'comp': 0.97, 'energy': -0.059*96485, 'points': []}
if not any(phase['name'] == 'α-Tb' for phase in bl.phases):
    bl.phases.insert(4, aTb_allotrope)
bl.phases[1]['energy'] = -0.336*96485

print("Solid phase energies (eV/atom):")
for p in bl.phases:
    if 'energy' in p:
        print(p['name'], round(p['energy']/96485, 3))
print("-" * 30)

measured_liq_points = [[p[0]/100.0, p[1]+273.15] for p in liquidus_points] # reformat to at. frac and Kelvin

# Given the behavior of the predicted liquidus around this composition range, there is likely a eutectic point between
solidus_midpoint = [(solidus_points[0][0] + solidus_points[1][0])/200.0, 
                   (solidus_points[0][1] + solidus_points[1][1])/2 + 273.15]
measured_liq_points.pop(-2)
measured_liq_points.append(solidus_midpoint)
measured_liq_points.sort()
# bl.invariants = [{'type': 'eut', 'comp': solidus_midpoint[0], 'temp': solidus_midpoint[1],
#                   'phases': [bl.phases[3]['name'], bl.phases[4]['name']],
#                   'phase_comps': [bl.phases[3]['comp'], bl.phases[4]['comp']]}]

adj_params = optimize_predicted_params(bl, measured_liq_points, ml_parameters)

fig = blp.get_plot('pred')
add_experimental_traces(fig)
fig.show()

Solid phase energies (eV/atom):
Pb 0.0
TbPb3 -0.336
Tb5Pb4 -0.542
Tb5Pb3 -0.543
α-Tb -0.059
Tb 0.0
------------------------------
Known points (Kelvin): [[0.1, 1130.9499999999998], [0.85, 1261.6], [0.9, 1419.15]]
Liquidus MAE from known points: 381K
Original predicted parameters: [-147186.4688, -18.38106271, 52796.44231, 0]

Maximum composition range fitted: [0.1, 0.9]
Ignored composition ranges: []

Initial triangle for pseudo-constraints: [[0.0, 0.0], [-55777.30505721702, 12609.665025604707], [-111554.61011443404, -12609.665025604707]]
--- Beginning Nelder-Mead optimization ---
Iteration # 0
T=0K enthalpy constraint violated for params [-250997.87275747658, -31.374734094684573, 0.0, 0.0]
Best guess: {a: -111554.61011443404, c: -12609.665025604707} f=518.754943598079
Height of triangle = 167331.91517165105
--- 0.3998236656188965 seconds elapsed ---
Iteration # 1
T=0K enthalpy constraint violated for params [-223109.22022886807, -27.88865252860851, -25219.330051209414, -3.1524162564011

Now, let's compare the updated phase diagram to our inital guess from the machine-learned parameters

In [28]:
manual_adjusted_params = [-111366.5242774636, -32.790766848153876, 26270.88383874484, 0.0]
bl.update_params(manual_adjusted_params)
# bl.update_params(adj_params) # update the parameters and generate the DFT-referenced phase diagram
bl.digitized_liq = predicted_liquidus_line # set the original predicted line as the liquidus reference for plotting
fig = blp.get_plot('pred+liq') # generate the plot as a figure object so it can be modified
add_experimental_traces(fig) # add the experimental data to the plot

# Manually edit trace colors and names and update layout. This may need to be altered if the trace order changes
fig.data[0].line.color = '#117733' # reference liquidus - plotted line 
fig.data[9].line.color = '#962EFF' # adjusted liquidus - plotted line
fig.data[10].line.color = '#117733' # reference liquidus - legend
fig.data[10].name = 'Original Liquidus Prediction'
fig.data[11].line.color = '#962EFF' # adjusted liquidus - legend
fig.data[11].name = 'Revised Liquidus Prediction'
fig.update_layout(width=900, height=700)
fig.show()

# Create another version without labels for manual labeling in powerpoint
fig_dict = fig.to_dict()
fig_dict['layout'].pop('title', None)
fig_dict['layout']['xaxis'].pop('title', None)
fig_dict['layout']['yaxis'].pop('title', None)
fig_dict['layout'].pop('annotations', None)
fig_no_labels = go.Figure(fig_dict)
fig_no_labels.layout.showlegend = False
fig_no_labels.layout.xaxis.showticklabels = False
fig_no_labels.layout.yaxis.showticklabels = False
fig_no_labels.update_layout(yaxis_range=[250, 1940])
fig_no_labels.add_trace(go.Scatter(
    x=[bl.phases[0]['comp']*100, bl.phases[1]['comp']*100],
    y=[bl.component_data[bl.components[0]][1]-275, bl.component_data[bl.components[0]][1]-275],
    mode='lines',
    line=dict(color='silver', width=4),
    showlegend=False))
fig_no_labels.show()

# Save the figure without labels
figures_dir = cfg.project_root / "figures"
if os.path.isdir(figures_dir):
    fig_no_labels.write_image(figures_dir / "Pb-Tb_phase_diagram_cleaned.svg")
    print(f"Figure saved to {figures_dir}")

[0, 25.0]
[327.55000000000007, 327.55000000000007]


Figure saved to c:\Users\willwerj\University of Michigan Dropbox\Joshua Willwerth\WHSun_Lab\G_liquid\gliquid_python\figures
